In [1]:
!pip install mamba-ssm --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 5.0 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 44.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 26.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 60.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 47.9 MB/s eta 0:00:00
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.2.post1-cp312-cp312-linux_x86_64.whl size=322288410 sha256=7a67070c1e7e99c95abd1319623f044e8a1b3fb46f774bfdea949f0a4fc79638
  Stored in directory: /root/.cache/pip/wheels/da/67/03/99148d6eeaa4ec2855d71295ac83bcbc8ba

In [4]:
!pip install causal-conv1d>=1.4.0 --no-build-isolation

In [5]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.0 MB/s eta 0:00:0000:01


In [16]:
import os
import wfdb
import torch
import torch.nn as nn
import numpy as np
from scipy.signal import resample
from torch.utils.data import Dataset, DataLoader, random_split
from torch.amp import autocast
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score, hamming_loss, accuracy_score
from mamba_ssm import Mamba
from tqdm.notebook import tqdm

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        norm_x = torch.mean(x ** 2, dim=-1, keepdim=True)
        x_normed = x * torch.rsqrt(norm_x + self.eps)
        return self.weight * x_normed

class BiMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.forward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.backward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        
    def forward(self, x):
        normed_x = self.norm(x)
        fwd_out = self.forward_mamba(normed_x)
        bwd_out = self.backward_mamba(normed_x.flip(dims=[1])).flip(dims=[1])
        return x + fwd_out + bwd_out

class AdvancedECGMamba(nn.Module):
    def __init__(self, in_channels=12, d_model=256, d_state=32, num_classes=5, num_layers=6):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, d_model // 2, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(d_model // 2),
            nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        self.layers = nn.ModuleList([BiMambaBlock(d_model=d_model, d_state=d_state) for _ in range(num_layers)])
        self.final_norm = RMSNorm(d_model)
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        x = self.stem(x) 
        x = x.transpose(1, 2)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        x_max, _ = torch.max(x, dim=1)
        return self.head(x_max)

def evaluate_mamba_advanced(model, dataloader, device):
    model.eval()
    all_probs, all_targets = [], []
    
    with torch.no_grad():
        for signals, labels in dataloader:
            signals = signals.to(device, non_blocking=True)
            with autocast('cuda', dtype=torch.bfloat16):
                probs = torch.sigmoid(model(signals)) 
            all_probs.append(probs.cpu().float().numpy())
            all_targets.append(labels.numpy())
            
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    
    # ---------------------------------------------------------
    # THE FIX: Calculate metrics per-class to safely ignore 
    # classes with 0 positive samples in the validation set.
    # ---------------------------------------------------------
    aucs = []
    auprcs = []
    superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    
    for i in range(len(superclasses)):
        # Check if this class has at least one '0' and at least one '1'
        if len(np.unique(all_targets[:, i])) == 2:
            aucs.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))
            auprcs.append(average_precision_score(all_targets[:, i], all_probs[:, i]))
            
    # Take the average ONLY across the valid classes
    macro_auc = np.mean(aucs) if len(aucs) > 0 else 0.0
    macro_auprc = np.mean(auprcs) if len(auprcs) > 0 else 0.0
        
    return macro_auc, macro_auprc

In [17]:
import os
import torch
import numpy as np
from scipy.io import loadmat
from scipy.signal import resample
from torch.utils.data import Dataset
from tqdm.notebook import tqdm

# ==========================================
# GEORGIA DATASET (NATIVE SCIPY LOADER)
# ==========================================
class GeorgiaDataset(Dataset):
    def __init__(self, data_path, target_hz=100):
        super().__init__()
        self.data_path = data_path
        self.superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
        self.target_length = target_hz * 10 # 10 seconds of data
        
        # Consistent SNOMED-CT Mapping to PTB-XL Superclasses
        self.snomed_mapping = {
            '426783006': 'NORM', '426177001': 'NORM', '427084000': 'NORM', '427393009': 'NORM',
            '164865005': 'MI', '164861001': 'MI', '54329005': 'MI',
            '164934002': 'STTC', '59931005': 'STTC', '164917005': 'STTC', '111975006': 'STTC',
            '164909002': 'CD', '733534002': 'CD', '59118001': 'CD', '713427006': 'CD', 
            '713426002': 'CD', '445118002': 'CD', '270492004': 'CD', '164947007': 'CD', '6374002': 'CD',
            '164873001': 'HYP', '370355005': 'HYP'
        }

        # Look for the actual .mat data files to guarantee we only process valid pairs
        self.files = [f[:-4] for f in os.listdir(data_path) if f.endswith('.mat') and not f.startswith('.')]
        
        self.signals = []
        self.labels = []
        self.skipped_files = 0
        
        print(f"Pre-loading Georgia database using SciPy and downsampling to {target_hz}Hz...")
        for f in tqdm(self.files, desc="Loading ECGs"):
            mat_path = os.path.join(self.data_path, f + '.mat')
            hea_path = os.path.join(self.data_path, f + '.hea')
            
            try:
                # 1. Read the signal bypassing WFDB
                mat_data = loadmat(mat_path)
                
                # PhysioNet CINC challenge stores the signal in a variable named 'val'
                if 'val' in mat_data:
                    signal = np.array(mat_data['val'], dtype=np.float32)
                else:
                    # Fallback if variable is named something else
                    key = [k for k in mat_data.keys() if not k.startswith('__')][0]
                    signal = np.array(mat_data[key], dtype=np.float32)
                
                # Ensure shape is (time, channels) for resampling
                if signal.shape[0] == 12:
                    signal = signal.T 
                
                # Standardize voltage scale to match PTB-XL (divide by 1000)
                signal = signal / 1000.0
                
                # Clean and Downsample
                signal = np.nan_to_num(signal)
                signal = resample(signal, self.target_length, axis=0) 
                
                # Convert back to (Channels, Time) for PyTorch Conv1d
                signal_tensor = torch.tensor(signal, dtype=torch.float32).transpose(0, 1)
                
                # 2. Read the header file manually, ignoring corrupted formatting/BOMs
                with open(hea_path, 'r', encoding='utf-8-sig', errors='ignore') as text_file:
                    lines = text_file.readlines()
                
                dx_str = ""
                for line in lines:
                    if line.startswith('#Dx:') or line.startswith('# Dx:'):
                        dx_str = line.split(':')[1].strip()
                        break
                
                # If no diagnosis is found, skip it
                if not dx_str:
                    self.skipped_files += 1
                    continue
                
                # Multi-hot encode the labels
                label = np.zeros(len(self.superclasses))
                for code in dx_str.split(','):
                    code = code.strip()
                    if code in self.snomed_mapping:
                        idx = self.superclasses.index(self.snomed_mapping[code])
                        label[idx] = 1
                        
                self.signals.append(signal_tensor)
                self.labels.append(torch.tensor(label, dtype=torch.float32))
                
            except Exception as e:
                self.skipped_files += 1
                continue

        print(f"Successfully loaded {len(self.signals)} records. Skipped {self.skipped_files} corrupted files.")

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

In [18]:
# 1. Setup Directories
GEORGIA_DIR = "/kaggle/input/datasets/organizations/physionet/georgia-12lead-ecg-challenge-database/Georgia"

# 2. Load and Split the Dataset (80% Train, 20% Validation)
full_dataset = GeorgiaDataset(data_path=GEORGIA_DIR, target_hz=100)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
generator = torch.Generator().manual_seed(42) # Seed ensures the split stays consistent

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

# num_workers=0 is required here to prevent Kaggle RAM crashing
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print(f"Training on {train_size} records. Validating on {val_size} records.")

# 3. Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AdvancedECGMamba(in_channels=12, d_model=256, d_state=32, num_classes=5, num_layers=6).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

# 4. The Training Loop with Live Validation
epochs = 30
max_norm = 1.0

print("Starting training from scratch on Georgia Database...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    
    for i, (signals, labels) in enumerate(train_dataloader):
        signals, labels = signals.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
        if torch.isnan(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        total_loss += loss.item()
        valid_batches += 1
        
    scheduler.step()
    avg_train_loss = total_loss / max(valid_batches, 1)
    
    # 5. Live Validation check at the end of each epoch
    val_auc, val_auprc = evaluate_mamba_advanced(model, val_dataloader, device)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val AUC: {val_auc:.4f} | Val AUPRC: {val_auprc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Optional: Save the weights when done
torch.save(model.state_dict(), '/kaggle/working/mamba_georgia_weights.pth')
print("Model saved to /kaggle/working/mamba_georgia_weights.pth")

Pre-loading Georgia database using SciPy and downsampling to 100Hz...


Loading ECGs:   0%|          | 0/10344 [00:00<?, ?it/s]

Successfully loaded 10344 records. Skipped 0 corrupted files.
Training on 8275 records. Validating on 2069 records.
Starting training from scratch on Georgia Database...
Epoch [1/30] | Train Loss: 0.3760 | Val AUC: 0.8601 | Val AUPRC: 0.7197 | LR: 0.000976
Epoch [2/30] | Train Loss: 0.3207 | Val AUC: 0.8877 | Val AUPRC: 0.7782 | LR: 0.000905
Epoch [3/30] | Train Loss: 0.2929 | Val AUC: 0.8999 | Val AUPRC: 0.7866 | LR: 0.000794
Epoch [4/30] | Train Loss: 0.2710 | Val AUC: 0.9170 | Val AUPRC: 0.8043 | LR: 0.000655
Epoch [5/30] | Train Loss: 0.2447 | Val AUC: 0.9179 | Val AUPRC: 0.8094 | LR: 0.000501
Epoch [6/30] | Train Loss: 0.2216 | Val AUC: 0.9236 | Val AUPRC: 0.8242 | LR: 0.000346
Epoch [7/30] | Train Loss: 0.1972 | Val AUC: 0.9246 | Val AUPRC: 0.8329 | LR: 0.000207
Epoch [8/30] | Train Loss: 0.1654 | Val AUC: 0.9245 | Val AUPRC: 0.8310 | LR: 0.000096
Epoch [9/30] | Train Loss: 0.1327 | Val AUC: 0.9214 | Val AUPRC: 0.8264 | LR: 0.000025
Epoch [10/30] | Train Loss: 0.1069 | Val AUC: 0